# 03. Inputs, CLI Runs, and Interactive Python API

This notebook answers the practical questions:

- What columns are required?
- Do I need to specify generations?
- Where do BAM and PLINK paths go?
- What does a minimal CLI run look like?
- What does a working interactive Python run look like?

We run a tiny slice of the synthetic benchmark so the tables and outputs shown below are real.


In [1]:
from pathlib import Path
import os, sys, json, math, shutil, time

REPO = Path('/home/bonnie/Documents/codex/STITCHV2')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
sys.path.insert(0, str(REPO / 'src'))
FIG_DIR = REPO / 'docs' / 'tutorial_deep' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = REPO / 'benchmark_runs' / 'synth_5mb_2k_0p1x'
OUT_DIR = REPO / 'benchmark_runs' / 'tutorial_deep_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)
print('repo:', REPO)
print('synthetic data exists:', DATA_DIR.exists())


repo: /home/bonnie/Documents/codex/STITCHV2
synthetic data exists: True


## Sample Table Requirements

The sample table can be parquet, CSV, or TSV-like text. It must contain:

| Column | Required? | Meaning | STITCH equivalent |
|---|---:|---|---|
| `generation` | yes | Per-sample transition scale, analogous to `nGen`. | `nGen`, but STITCH has one value for the run. |
| `sample_id` | recommended | Sample name used in outputs and matching PLINK IID. If absent, STITCHV2 creates `sample_0`, ... | `sampleNames_file` |
| `bam_path` | optional/fillable | BAM/CRAM path. Empty string means no reads for that sample. | `bamlist` or `cramlist` |
| `sex` | optional | Used only with `--ploidy-males/--ploidy-females`. | no direct equivalent |
| `plink_path` | optional | Per-sample PLINK prefix for microarray hard-call evidence. | closest: `genfile`, but STITCH format differs |

Yes: you need `generation`. If you do not know it, choose a reasonable shared value for all samples and tune it by CV/benchmarking.


In [2]:
samples = pd.read_parquet(DATA_DIR / 'samples.parquet')
positions = pd.read_parquet(DATA_DIR / 'positions.parquet')
print('samples shape:', samples.shape)
display(samples.head())
display(samples.tail())
print('positions shape:', positions.shape)
display(positions.head())
display(positions.tail())
print('required generation present:', 'generation' in samples.columns)
print('first BAM exists:', Path(samples.loc[0, 'bam_path']).exists())


samples shape: (48, 3)
    sample_id                                           bam_path  generation
0  sample_000  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0
1  sample_001  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0
2  sample_002  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0
3  sample_003  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0
4  sample_004  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0
     sample_id                                           bam_path  generation
43  sample_043  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0
44  sample_044  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0
45  sample_045  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0
46  sample_046  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0
47  sample_047  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0
positions shape: (2000, 4)
            CHR   PO

## Position Table Requirements

The position table must contain `CHR` and `POS`; real runs should also provide `REF` and `ALT`.

Original STITCH `posfile` is a headerless text file with chromosome, position, reference, alternate. STITCHV2 accepts that information in parquet or CSV/TSV-style tables.


## PLINK Inputs

STITCHV2 can use PLINK in two ways:

1. `--founder-plink founder_prefix`: initialize founders from a reference/founder PLINK BED/BIM/FAM set.
2. `--microarray-plink array_prefix` or `samples.parquet: plink_path`: inject hard genotypes as microarray evidence.

For `plink_path`, several samples may point to the same PLINK prefix. STITCHV2 loads each unique prefix once, extracts only matching `sample_id` values, aligns variants by chromosome/position, then injects hard calls using `--microarray-hard-call-weight`.


In [3]:
example = samples.head(6).copy()
example['sex'] = ['M', 'F', 'M', 'F', 'XY', 'XX']
example['plink_path'] = ['arrays/batch1'] * 3 + ['arrays/batch2'] * 3
print('Example sample table with optional sex and per-sample PLINK paths:')
display(example)


Example sample table with optional sex and per-sample PLINK paths:
    sample_id                                           bam_path  generation sex     plink_path
0  sample_000  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0   M  arrays/batch1
1  sample_001  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0   F  arrays/batch1
2  sample_002  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0   M  arrays/batch1
3  sample_003  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0   F  arrays/batch2
4  sample_004  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0  XY  arrays/batch2
5  sample_005  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0  XX  arrays/batch2


## Minimal CLI Example

```bash
python -m pip install -e . --no-build-isolation
stitchv2 run --samples benchmark_runs/synth_5mb_2k_0p1x/samples.parquet --positions benchmark_runs/synth_5mb_2k_0p1x/positions.parquet --chromosome chrSynthetic --output-dir benchmark_runs/tutorial_cli_run --n-founders 8 --em-iterations 5 --block-size 1000 --hmm-backend jax --write-genotype-posteriors --write-genotype-calls
```

Equivalent STITCH concepts:

```r
STITCH::STITCH(chr='chrSynthetic', posfile='pos.txt', bamlist='bamlist.txt', K=8, nGen=10, iterations=5, outputdir='stitch_out')
```


## Working Interactive Python Example

The Python API is useful when you already have pandas tables, an in-memory founder panel, or a pedigree matrix. The only method you call is `StitchPipeline(config).prepare_inputs(...)`; despite the name, this performs the full block run and writes outputs.


In [4]:
from stitchv2.config import PipelineConfig
from stitchv2.founders import FounderPanel
from stitchv2.pipeline import StitchPipeline

run_base = OUT_DIR / '03_interactive_python'
if run_base.exists():
    shutil.rmtree(run_base)
run_base.mkdir(parents=True)

samples_small = samples.head(6).copy()
positions_small = positions.head(60).copy()
positions_path = run_base / 'positions.parquet'
samples_path = run_base / 'samples.parquet'
positions_small.to_parquet(positions_path, index=False)
samples_small.to_parquet(samples_path, index=False)

founder_panel = FounderPanel(
    chromosome='chrSynthetic',
    positions=positions_small['POS'].to_numpy(dtype=np.int64),
    ref=positions_small['REF'].astype(str).to_numpy(),
    alt=positions_small['ALT'].astype(str).to_numpy(),
    alt_prob=np.full((4, len(positions_small)), 0.5, dtype=np.float32),
    immutable_mask=np.zeros(4, dtype=bool),
)

config = PipelineConfig(
    chromosome='chrSynthetic',
    positions_path=positions_path,
    output_dir=run_base / 'run',
    n_founders=4,
    block_size=30,
    em_iterations=1,
    hmm_backend='numpy',
    read_mode='read_stream',
    read_stream_backend='python',
    write_genotype_posteriors=True,
    write_genotype_calls=True,
    write_support_mask=True,
    write_transitions=False,
    profile_memory=True,
    random_seed=3,
)

start = time.perf_counter()
StitchPipeline(config).prepare_inputs(samples_small, founder_panel=founder_panel)
elapsed = time.perf_counter() - start
print(f'interactive run elapsed: {elapsed:.3f} seconds')
print('output files:', sorted(p.name for p in (run_base / 'run').iterdir()))


interactive run elapsed: 0.324 seconds
output files: ['dosage', 'founder_updates', 'founders.parquet', 'genotype_calls', 'genotype_posteriors', 'memory_profile_summary.json', 'positions.parquet', 'recombination', 'samples.parquet', 'stage_timings.json', 'support_mask']


In [5]:
run_dir = run_base / 'run'
print('Written samples with resolved ploidy:')
display(pd.read_parquet(run_dir / 'samples.parquet').head())

print('Dosage head/tail:')
dosage = pd.read_parquet(run_dir / 'dosage')
display(dosage.head())
display(dosage.tail())

print('Genotype posterior head:')
gp = pd.read_parquet(run_dir / 'genotype_posteriors')
display(gp.head())

print('Memory summary:')
print(json.dumps(json.loads((run_dir / 'memory_profile_summary.json').read_text()), indent=2))


Written samples with resolved ploidy:
    sample_id                                           bam_path  generation  ploidy
0  sample_000  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0       2
1  sample_001  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0       2
2  sample_002  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0       2
3  sample_003  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0       2
4  sample_004  /home/bonnie/Documents/codex/STITCHV2/benchmar...        10.0       2
Dosage head/tail:
    sample_id    chromosome  position  dosage  block_id
0  sample_000  chrSynthetic       239     1.0         0
1  sample_000  chrSynthetic       851     1.0         0
2  sample_000  chrSynthetic      3036     1.0         0
3  sample_000  chrSynthetic      3312     1.0         0
4  sample_000  chrSynthetic      6659     1.0         0
      sample_id    chromosome  position  dosage  block_id
355  sample_005  chrSynthetic    155623 

## Output Format Recap

- `dosage/`: one row per sample and variant with expected alternate allele count.
- `genotype_posteriors/`: list column of length `P + 1`, representing alt allele count `0..P`.
- `genotype_calls/`: hard call integer `0..P`, or `-1` for no-call modes.
- `support_mask/`: direct evidence flag.
- `stage_timings.json`: read/HMM/calibration/write timing and memory by block.
- `memory_profile_summary.json`: peak RSS summary.

Later notebooks show calibration plots, Dask reports, and PLINK conversion.
